# Neuron Clustering

This notebook clusters neurons by subclass using cosine-distance k-means on MLP activations.

**Requirements:**
- Place your `epoch_4000.pt` checkpoint in the Colab working directory
- Run all cells in order

## 1. Install Dependencies

In [ ]:
!pip install -q transformers accelerate huggingface_hub

## 2. HuggingFace Login

Required for Llama model access.

In [ ]:
from huggingface_hub import login

# Option 1: Try to get token from Colab secrets
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print('Logged in using Colab secret HF_TOKEN')
    else:
        login()  # Interactive login
except:
    # Option 2: Interactive login
    login()

## 3. Imports and Setup

In [ ]:
import json
import os
import random

import torch
import torch.nn.functional as F
from torch import nn
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Create directories
os.makedirs('datasets', exist_ok=True)
os.makedirs('results/neuron-clustering', exist_ok=True)
os.makedirs('results/activations', exist_ok=True)

## 4. Configuration

In [ ]:
# Configuration - modify these as needed
model_name = "meta-llama/Llama-3.2-1B"  # or "meta-llama/Meta-Llama-3-8B"
checkpoint_path = "epoch_4000.pt"  # Path to your checkpoint file
k_classes = 8
lr = 1e-3
threshold = 1e-3

# Model configs
llama_1b = "meta-llama/Llama-3.2-1B"
llama_8b = "meta-llama/Meta-Llama-3-8B"

config = {
    "1b": AutoConfig.from_pretrained(llama_1b),
    "8b": AutoConfig.from_pretrained(llama_8b),
}

## 5. Generate Dataset

Generate all 2-digit addition pairs programmatically.

In [ ]:
def gen_2d_add_dataset(dataset_fname, tokenizer):
    """Generate all 2-digit addition pairs (10000 total)."""
    all_pairs = [(f'{num1}+{num2}=', num1 + num2) for num1 in range(100) for num2 in range(100)]
    random.shuffle(all_pairs)

    dataset = []
    for prompt, answer in all_pairs:
        q_str = prompt
        a_str = str(answer)
        ids = tokenizer.encode(q_str + a_str, add_special_tokens=False)
        dataset.append({
            "q_str": q_str,
            "a_str": a_str,
            "ids": ids,
        })

    with open(dataset_fname, 'w') as f:
        json.dump(dataset, f, indent=4)
    
    print(f"Generated {len(dataset)} examples -> {dataset_fname}")
    return dataset

# Generate the dataset
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

dataset_path = 'datasets/2d_add_all.json'
if not os.path.exists(dataset_path):
    gen_2d_add_dataset(dataset_path, tokenizer)
else:
    print(f"Dataset already exists at {dataset_path}")

## 6. Model Classes

In [ ]:
class ProblemEncoder(nn.Module):
    """Encodes (op1, op2, result) into a fixed-size embedding."""
    
    def __init__(self, embedding_dim):
        super().__init__()
        self.op1_emb_layer = nn.Embedding(100, embedding_dim // 4)
        self.op2_emb_layer = nn.Embedding(100, embedding_dim // 4)
        self.sum_emb_layer = nn.Embedding(200, embedding_dim // 2)

    def forward(self, op1, op2, res):
        op1_emb = self.op1_emb_layer(op1)
        op2_emb = self.op2_emb_layer(op2)
        sum_emb = self.sum_emb_layer(res)
        return torch.cat((op1_emb, op2_emb, sum_emb), dim=-1)


class ProblemClassifier(nn.Module):
    """MLP classifier that maps problem encoding to k class logits."""
    
    def __init__(self, input_dim, k_classes, hidden1_dim=256, hidden2_dim=32):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden1_dim),
            nn.ReLU(),
            nn.Linear(hidden1_dim, hidden2_dim),
            nn.ReLU(),
            nn.Linear(hidden2_dim, k_classes),
        )

    def forward(self, x):
        return self.classifier(x)


class NeuronMask(nn.Module):
    """Learnable soft masks over MLP neurons."""
    
    def __init__(self, k_classes, activations_dim):
        super().__init__()
        hidden_dim = 4
        self.k_classes = k_classes
        self.class_embedding = nn.Embedding(k_classes, hidden_dim)
        self.output_layer = nn.Linear(hidden_dim, activations_dim)

    def forward(self, class_probs, activations):
        class_ids = class_probs.argmax(dim=-1)
        hidden = self.class_embedding(class_ids)
        hidden = F.relu(hidden)
        selected_mask = self.output_layer(hidden)
        sigmoid_mask = torch.sigmoid(selected_mask)
        sigmoid_mask_expanded = sigmoid_mask.unsqueeze(1)
        masked_activations = activations * sigmoid_mask_expanded
        return masked_activations, sigmoid_mask

    def class_masks(self):
        """Get the sigmoid masks for all classes."""
        device = self.class_embedding.weight.device
        class_ids = torch.arange(self.k_classes, device=device)
        hidden = self.class_embedding(class_ids)
        hidden = F.relu(hidden)
        masks = self.output_layer(hidden)
        return torch.sigmoid(masks)


class CircuitDiscoveryModel(nn.Module):
    """Wrapper holding neuron masks and problem classifier for circuit discovery."""
    
    def __init__(self, k_classes, problem_embedding_dim=256, tau=0.5):
        super().__init__()
        self.tau = tau
        
        num_activations_1b = config["1b"].intermediate_size * config["1b"].num_hidden_layers
        num_activations_8b = config["8b"].intermediate_size * config["8b"].num_hidden_layers

        self.problem_encoder = ProblemEncoder(embedding_dim=problem_embedding_dim)
        self.classifier = ProblemClassifier(problem_embedding_dim, k_classes)

        self.neuron_masks_1b = NeuronMask(k_classes, num_activations_1b)
        self.neuron_masks_8b = NeuronMask(k_classes, num_activations_8b)

    def classify_problem(self, op1, op2, res):
        problem_encoding = self.problem_encoder(op1, op2, res)
        logits = self.classifier(problem_encoding)
        return logits

    def forward(self, op1, op2, res, activations_1b, activations_8b):
        logits = self.classify_problem(op1, op2, res)
        hard_class_probs = F.gumbel_softmax(logits, tau=self.tau, hard=True, dim=-1)
        masked_activations_1b, mask_1b = self.neuron_masks_1b(hard_class_probs, activations_1b)
        masked_activations_8b, mask_8b = self.neuron_masks_8b(hard_class_probs, activations_8b)
        return {
            "hard_class_probs": hard_class_probs,
            "masked_activations_1b": masked_activations_1b,
            "masked_activations_8b": masked_activations_8b,
            "mask_1b": mask_1b,
            "mask_8b": mask_8b,
        }

## 7. Utility Functions

In [ ]:
def _safe_model_name(model_name: str) -> str:
    return model_name.replace("/", "_").replace(":", "_")


def _stack_layer_activations(batch_activations):
    if not batch_activations:
        raise ValueError("batch_activations is empty")
    layers = sorted(batch_activations.keys())
    tensors = [batch_activations[i] for i in layers]
    return torch.cat(tensors, dim=-1)


def parse_equation(probs, device=None):
    op1_list, op2_list, res_list = [], [], []
    for prob in probs:
        add_idx = prob.index("+")
        equal_idx = prob.index("=")
        op1_str = prob[:add_idx]
        op2_str = prob[add_idx + 1 : equal_idx]
        res_str = prob[equal_idx + 1 :]
        op1_list.append(int(op1_str))
        op2_list.append(int(op2_str))
        res_list.append(int(res_str))
    op1 = torch.tensor(op1_list, dtype=torch.long, device=device)
    op2 = torch.tensor(op2_list, dtype=torch.long, device=device)
    res = torch.tensor(res_list, dtype=torch.long, device=device)
    return op1, op2, res


def load_model_checkpoint(checkpoint_path, k_classes, lr, problem_embedding_dim=256, tau=0.5):
    """Load a CircuitDiscoveryModel from checkpoint."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = CircuitDiscoveryModel(
        k_classes=k_classes,
        problem_embedding_dim=problem_embedding_dim,
        tau=tau
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    epoch = checkpoint.get("epoch", checkpoint.get("step", 0))
    metrics_log = checkpoint.get("metrics_log", [])
    return model, optimizer, metrics_log, epoch

## 8. Neuron Activations Generator

In [ ]:
class NeuronActivationsGenerator:
    """Generate MLP neuron activations for the 2-digit addition dataset."""

    def __init__(self, model_name, batch_size=50):
        self.llm = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else None,
        ).to(device)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        self.llm.eval()
        self.model_name = model_name
        self.safe_model_name = model_name.replace('/', '_').replace(':', '_')
        self.batch_size = batch_size

        with open('datasets/2d_add_all.json', 'r') as f:
            dataset = json.load(f)

        ids = [record['ids'] for record in dataset]
        self.ids = torch.tensor(ids).to(self.llm.device)
        self.layer_activations = {}

        self.handles = []
        for i, layer in enumerate(self.llm.model.layers):
            h = layer.mlp.up_proj.register_forward_hook(self.make_hook(i))
            self.handles.append(h)

    def make_hook(self, layer_idx):
        def hook(module, inputs, output):
            activ = output[:, :, :].detach().cpu()
            self.layer_activations.setdefault(layer_idx, []).append(activ)
        return hook
    
    def generate_batch_activations(self, batch, log=True):
        with torch.no_grad():
            start_prob = batch * self.batch_size
            batch_inputs = self.ids[start_prob: start_prob + min(self.batch_size, self.ids.shape[0] - start_prob)]
            
            if log: 
                print(f'Processing batch {batch}/{self.ids.shape[0] // self.batch_size}')

            self.layer_activations = {}
            _ = self.llm(input_ids=batch_inputs)
            batch_activations = {}
            for layer_idx, chunks in self.layer_activations.items():
                batch_activations[layer_idx] = torch.cat(chunks, dim=0)
            
            activations = {
                'ids': batch_inputs,
                'activations': batch_activations,
            }

            out_dir = 'results/activations'
            os.makedirs(out_dir, exist_ok=True)
            out_fname = os.path.join(out_dir, f'activations_{self.safe_model_name}_{batch}.pt')
            torch.save(activations, out_fname)
            return out_fname

    def remove_handles(self):
        for h in self.handles:
            h.remove()

## 9. Load Circuit Discovery Model

In [ ]:
# Load the circuit discovery model from checkpoint
model, optimizer, metrics_log, epoch = load_model_checkpoint(
    checkpoint_path, 
    k_classes=k_classes, 
    lr=lr
)
model.eval()
print(f"Loaded checkpoint from epoch {epoch}")

## 10. Extract Neuron Masks

In [ ]:
# Get neuron masks based on model
if model_name == "meta-llama/Llama-3.2-1B":
    neuron_masks = model.neuron_masks_1b.class_masks()
else:
    neuron_masks = model.neuron_masks_8b.class_masks()

neuron_masks = neuron_masks > (1 - threshold)

print(f"Active neurons ratio: {torch.mean(torch.mean(neuron_masks.float(), dim=1)).item():.4f}")
print(f"\nNeurons per subclass:")
for i in range(k_classes):
    count = neuron_masks[i].count_nonzero().item()
    print(f"  Subclass {i}: {count} neurons")

## 11. K-Means Clustering (Cosine Distance)

In [ ]:
def _kmeans_cosine(x, k, num_iters=20):
    """
    Balanced k-means clustering using cosine distance.
    
    Args:
        x: Input tensor of shape (N, D)
        k: Number of clusters
        num_iters: Maximum number of iterations
    
    Returns:
        cluster_ids: Cluster assignment for each point
        centroids: Final cluster centroids
        loss: Mean cosine distance to assigned centroids
    """
    N, D = x.shape
    if k > N:
        raise ValueError("k cannot be larger than number of points")

    x = F.normalize(x, p=2, dim=-1, eps=1e-8)

    # k-means++ initialization
    indices = []
    first = torch.randint(0, N, (1,), device=x.device)
    indices.append(first.item())
    for _ in range(1, k):
        centers = x[torch.tensor(indices, device=x.device)]
        sim = x @ centers.t()
        closest_sim, _ = sim.max(dim=1)
        dist = 1 - closest_sim.clamp(-1, 1)
        probs = dist / dist.sum()
        next_idx = torch.multinomial(probs, 1)
        indices.append(next_idx.item())

    centroids = x[torch.tensor(indices, device=x.device)]

    # Balanced assignment capacities
    base_cap = N // k
    remainder = N % k
    capacities = torch.full((k,), base_cap, device=x.device, dtype=torch.long)
    if remainder > 0:
        capacities[:remainder] += 1

    prev_cluster_ids = None
    prev_loss = None
    loss = None

    for iter_num in range(num_iters):
        sim = x @ centroids.t()
        dists = 1.0 - sim.clamp(-1.0, 1.0)

        cluster_ids = torch.full((N,), -1, device=x.device, dtype=torch.long)
        remaining_cap = capacities.clone()

        _, sorted_clusters = torch.sort(dists, dim=1)

        for rank in range(k):
            unassigned = cluster_ids.eq(-1)
            if not unassigned.any():
                break

            cand_clusters = sorted_clusters[unassigned, rank]
            unassigned_idx = unassigned.nonzero(as_tuple=False).squeeze(1)

            for j in range(k):
                if remaining_cap[j] <= 0:
                    continue

                want_j_mask = cand_clusters.eq(j)
                if not want_j_mask.any():
                    continue

                cand_indices = unassigned_idx[want_j_mask]
                take = min(remaining_cap[j].item(), cand_indices.numel())
                if take <= 0:
                    continue

                chosen = cand_indices[:take]
                cluster_ids[chosen] = j
                remaining_cap[j] -= take

        if (cluster_ids == -1).any():
            raise RuntimeError("Balanced k-means assignment failed: some points unassigned")

        if prev_cluster_ids is not None and torch.equal(cluster_ids, prev_cluster_ids):
            break

        point_sim = sim[torch.arange(N, device=x.device), cluster_ids]
        point_dists = 1.0 - point_sim.clamp(-1.0, 1.0)
        loss = point_dists.mean().item()

        if prev_loss is not None and loss is not None:
            if abs(loss - prev_loss) < 1e-6:
                break

        prev_cluster_ids = cluster_ids.clone()
        prev_loss = loss

        # Update centroids
        new_centroids = torch.zeros_like(centroids)
        for j in range(k):
            mask = cluster_ids == j
            if mask.any():
                new_centroids[j] = x[mask].mean(dim=0)
            else:
                rand_idx = torch.randint(0, N, (1,), device=x.device)
                new_centroids[j] = x[rand_idx]

        centroids = F.normalize(new_centroids, p=2, dim=-1, eps=1e-8)

    return cluster_ids, centroids, loss

## 12. Collect Neuron Features Per Subclass

In [ ]:
def _collect_neuron_features_per_subclass(circuit_model, tokenizer, neuron_masks, model_name, batch_size=5, save_path=None):
    """
    Collect neuron activation features grouped by subclass.
    """
    activations_generator = NeuronActivationsGenerator(model_name, batch_size=batch_size)
    num_batches = (activations_generator.ids.shape[0] + batch_size - 1) // batch_size

    k_classes = neuron_masks.size(0)

    indices_per_subclass = {}
    for c in range(k_classes):
        mask = neuron_masks[c]
        idx = torch.nonzero(mask, as_tuple=False).squeeze(1)
        if idx.numel() == 0:
            continue
        indices_per_subclass[c] = idx

    features_lists = {c: [] for c in indices_per_subclass.keys()}

    for batch_idx in range(num_batches):
        out_fname = activations_generator.generate_batch_activations(batch_idx, log=True)
        batch = torch.load(out_fname, map_location="cpu")

        ids, activations_dict = batch["ids"], batch["activations"]

        if isinstance(ids, torch.Tensor):
            input_id_list = ids.tolist()
        else:
            input_id_list = ids

        prompts = tokenizer.batch_decode(input_id_list, skip_special_tokens=True)
        activations = _stack_layer_activations(activations_dict).to(device)

        op1, op2, res = parse_equation(prompts, device=device)
        classifier_logits = circuit_model.classify_problem(op1, op2, res)
        hard = F.gumbel_softmax(classifier_logits, tau=circuit_model.tau, dim=-1, hard=True)
        subclass = hard.argmax(dim=-1)

        mean_activations = activations.mean(dim=1)

        for c, idx in indices_per_subclass.items():
            ex_mask = subclass == c
            if not ex_mask.any():
                continue
            acts_c = mean_activations[ex_mask][:, idx]
            file_feature_c = acts_c.mean(dim=0)
            features_lists[c].append(file_feature_c)

    activations_generator.remove_handles()

    features_per_subclass = {}
    for c, feats in features_lists.items():
        if not feats:
            continue
        feats_tensor = torch.stack(feats, dim=0).to(device)
        features_per_subclass[c] = feats_tensor.t()

    if save_path is not None:
        torch.save(
            {
                "model_name": model_name,
                "features_per_subclass": {c: v.detach().cpu() for c, v in features_per_subclass.items()},
                "indices_per_subclass": {c: idx.detach().cpu() for c, idx in indices_per_subclass.items()},
            },
            save_path,
        )
        print(f"Saved subclass neuron features to {save_path}")

    return features_per_subclass, indices_per_subclass

## 13. Run Neuron K-Means Clustering

In [ ]:
def run_neuron_kmeans(
    k,
    subclass: int,
    circuit_model,
    tokenizer,
    neuron_masks,
    model_name,
    batch_size=5,
    num_iters=100,
    log=True,
    subclass_features_path=None,
):
    """
    Run k-means clustering on neurons for a specific subclass.
    """
    safe = _safe_model_name(model_name)
    results_dir = os.path.join("results", "neuron-clustering", safe)
    os.makedirs(results_dir, exist_ok=True)

    if subclass_features_path is None:
        subclass_features_path = os.path.join(results_dir, "subclass_features.pt")

    if subclass_features_path is not None and os.path.exists(subclass_features_path):
        ckpt = torch.load(subclass_features_path, map_location=device)
        features_per_subclass = {int(c): v.to(device) for c, v in ckpt["features_per_subclass"].items()}
        indices_per_subclass = {int(c): idx.to(device) for c, idx in ckpt["indices_per_subclass"].items()}
        if log:
            print(f"Loaded cached features from {subclass_features_path}")
    else:
        features_per_subclass, indices_per_subclass = _collect_neuron_features_per_subclass(
            circuit_model, tokenizer, neuron_masks, model_name,
            batch_size=batch_size, save_path=subclass_features_path
        )

    if subclass not in features_per_subclass:
        raise ValueError(f"No features found for subclass {subclass}")

    x = features_per_subclass[subclass]
    subclass_indices = indices_per_subclass[subclass]

    cluster_ids, centroids, loss = _kmeans_cosine(x, k=k, num_iters=num_iters)

    cluster_to_indices = {}
    for j in range(k):
        mask = cluster_ids == j
        if mask.any():
            cluster_to_indices[j] = subclass_indices[mask].cpu()
        else:
            cluster_to_indices[j] = torch.empty(0, dtype=subclass_indices.dtype)

    # Save results
    clusters_path = os.path.join(results_dir, f"subclass_{subclass}_clusters", f"k{k}.pt")
    os.makedirs(os.path.dirname(clusters_path), exist_ok=True)
    torch.save(
        {
            "model_name": model_name,
            "subclass": subclass,
            "k": k,
            "cluster_ids": cluster_ids.cpu(),
            "subclass_indices": subclass_indices.cpu(),
            "cluster_to_indices": cluster_to_indices,
            "loss": loss,
        },
        clusters_path,
    )

    if log:
        print(f"Subclass {subclass}: k-means over neurons completed.")
        print(f"Mean cosine distance to centroids (loss): {loss:.6f}")
        for j in range(k):
            size = int((cluster_ids == j).sum().item())
            print(f"  Cluster {j}: size={size}")
        print(f"Saved cluster assignments to {clusters_path}")

    return cluster_ids, centroids, loss

## 14. Run Clustering on a Single Subclass

Example: cluster neurons for subclass 0 with k=7 clusters.

In [ ]:
# Run clustering for a single subclass
subclass = 0  # Change this to cluster different subclasses
k = 7  # Number of clusters

if neuron_masks[subclass].any().item():
    cluster_ids, centroids, loss = run_neuron_kmeans(
        k=k,
        subclass=subclass,
        circuit_model=model,
        tokenizer=tokenizer,
        neuron_masks=neuron_masks,
        model_name=model_name,
        batch_size=5,
        num_iters=100,
        log=True
    )
else:
    print(f"Subclass {subclass} has no active neurons")

## 15. Grid Search Over k Values

Run k-means for multiple values of k to find optimal clustering.

In [ ]:
# Grid search over k values for all subclasses
k_range = range(2, 10)  # Test k from 2 to 9
k_gs_results = {}

for subclass in range(k_classes):
    if neuron_masks[subclass].any().item():
        print(f"\nProcessing subclass {subclass}")
        k_gs_results[subclass] = {}
        for k in k_range:
            try:
                _, _, loss = run_neuron_kmeans(
                    k=k,
                    subclass=subclass,
                    circuit_model=model,
                    tokenizer=tokenizer,
                    neuron_masks=neuron_masks,
                    model_name=model_name,
                    log=False
                )
                k_gs_results[subclass][k] = loss
                print(f"  k={k}, loss={loss:.6f}")
            except ValueError as e:
                print(f"  k={k} failed: {e}")
    else:
        print(f"\nSkipping subclass {subclass} (no active neurons)")

## 16. Save Grid Search Results

In [ ]:
# Save grid search results to JSON
safe = _safe_model_name(model_name)
results_dir = os.path.join("results", "neuron-clustering", safe)
os.makedirs(results_dir, exist_ok=True)

out_path = os.path.join(results_dir, "k_gs_results.json")

# Convert to JSON-serializable format
k_gs_json = {str(k): {str(kk): v for kk, v in vv.items()} for k, vv in k_gs_results.items()}

with open(out_path, "w") as f:
    json.dump(k_gs_json, f, indent=2)

print(f"Saved grid search results to {out_path}")

## 17. Visualize Clustering Results

In [ ]:
import matplotlib.pyplot as plt

# Plot loss vs k for each subclass
fig, ax = plt.subplots(figsize=(10, 6))

for subclass, results in k_gs_results.items():
    ks = sorted(results.keys())
    losses = [results[k] for k in ks]
    ax.plot(ks, losses, marker='o', label=f'Subclass {subclass}')

ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Mean Cosine Distance (Loss)')
ax.set_title('Neuron Clustering: Loss vs Number of Clusters')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'k_vs_loss.png'), dpi=150)
plt.show()

## 18. Download Results (Optional)

In [ ]:
# Download results
from google.colab import files

# Zip the results folder
!zip -r results.zip results/

# Download
files.download('results.zip')